# CGR3 CUDA13 compatibility check

One proposed T4 check capped at120controlled seconds; setup/upload/export/idle extra.
Two training updates on one TRAIN cube5 example, then restore and replay update2:
three optimizer steps total. Includes augmented start and exact full-payload check.
No quality trial, no full256-update job, no automatic continuation or retry.
Expected observed stack: T4,Python3.13.15,Torch2.11.0+cu130,NumPy2.1.3,CUDA13.0,
cuDNN92700. Strict deterministic algorithms remain on. Changed stack stops again.
Open notebook,upload supplied ZIP,leave APPROVED_SEED_JOB=False until explicit
approval of this check. After approval set True and run once. Download ZIP+receipt,
including failure,then return both. Local download only,no Drive access.
This new package preserves the old training script unchanged; do not run it.
A successful check does not establish numerical equivalence with CUDA12.8 or
approve another training attempt. No checkpoint from this check is a quality model.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, zipfile, json, uuid, subprocess, sys
EXPECTED_SHA256 = 'd96ef7c05ea102b54841d4298d27d98f1f7f0701c29da459e26757500f86a3a9'
ARCHIVE_NAME = 'NCA-CGR3-cu130-Preflight-Package.zip'
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Select exactly the supplied CGR3 ZIP')
raw = next(iter(uploaded.values()))
if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256: raise ValueError('Package checksum differs')
source = Path('/content') / ('cgr3-compat-upload-' + uuid.uuid4().hex + '.zip')
with source.open('xb') as f: f.write(raw)
PACKAGE = Path('/content') / ('nca-cgr3-compat-' + uuid.uuid4().hex)
with zipfile.ZipFile(source) as z:
    manifest = json.loads(z.read('manifest.json'))
    names = z.namelist()
    if len(names) != len(set(names)) or set(names) != set(manifest['files']) | {'manifest.json'}: raise ValueError('Unexpected members')
    if sum(x.file_size for x in z.infolist()) > 100_000_000: raise ValueError('Expanded size limit')
    for name in names:
        p = PurePosixPath(name)
        if p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or str(p) != name: raise ValueError('Unsafe path')
        if ((z.getinfo(name).external_attr >> 16) & 0o170000) == 0o120000: raise ValueError('Symlink member')
        if name != 'manifest.json' and hashlib.sha256(z.read(name)).hexdigest() != manifest['files'][name]: raise ValueError('Member hash differs')
    PACKAGE.mkdir()
    for name in names:
        target = PACKAGE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as f: f.write(z.read(name))
subprocess.run([sys.executable, '-c', 'from nca.curriculum_package import verify; verify(".")'], cwd=PACKAGE, check=True)
print('Package verified:', PACKAGE)


In [ ]:
MODEL_SEED = 1201  # Only seed1201 is proposed.
APPROVED_SEED_JOB = False
if not APPROVED_SEED_JOB: raise RuntimeError('Stop until this seed job and backup plan are approved')
result = subprocess.run([sys.executable, str(PACKAGE / 'scripts/colab_curriculum_compatibility.py'),
    '--seed', str(MODEL_SEED), '--device', 'cuda:0', '--approved-seed-job', '--seconds', '120'], cwd=PACKAGE)
print('Exit code:', result.returncode, '- download evidence next, including failures.')


In [ ]:
exports = sorted((PACKAGE / 'compatibility-runs').glob('*.zip'))
if not exports: raise RuntimeError('No export. Preserve runtime and send the error; do not rerun.')
for archive in exports:
    files.download(str(archive))
    files.download(str(archive.with_suffix('.receipt.json')))
print('Return ZIP and receipt for local verification; disconnect idle GPU after verified download.')
print('No additional seed or automatic retry is planned.')
